# Morphology descriptors: t-SNE

- **Data** — the same standardised descriptors over the same labelled cells the figure uses, read
  from the `case_study_3/` session so the two cannot drift apart
- **Gives** — the projection panel **f** draws: t-SNE at perplexity 30 / 1000 iterations / seed 42
- **Kernel** — `mycol_colonies_env`, run top to bottom

In [ ]:
import io
import zipfile
from pathlib import Path

import pandas as pd
import plotly.express as px
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler

SESSION_ZIP = Path.cwd().parents[3] / "case_study_3" / "mycol_saved_session_CS3.zip"

with zipfile.ZipFile(SESSION_ZIP) as z:
    metrics = pd.read_csv(io.BytesIO(z.read("cell_metrics.csv")))
labelled = metrics[metrics["mask label"] != "Unlabelled"].reset_index(drop=True)

ID_COLS = {"image #", "image", "mask #", "mask label"}
DESCRIPTORS = [c for c in labelled.columns if c not in ID_COLS]

X = StandardScaler().fit_transform(labelled[DESCRIPTORS])
labels = labelled["mask label"]
COLOURS = {"normal": "#5289C7", "abnormal": "#4EB265"}   # as Figure 4

print(f"{len(labelled)} labelled cells of {len(metrics)}, {len(DESCRIPTORS)} descriptors")
print("  " + ", ".join(DESCRIPTORS))
print("  " + ", ".join(f"{k} {v}" for k, v in labels.value_counts().items()))

In [ ]:
def show(coords, x, y, title):
    fig = px.scatter(pd.DataFrame(coords, columns=[x, y]).assign(label=labels.values),
                     x=x, y=y, color="label", color_discrete_map=COLOURS,
                     title=title, template="plotly_white")
    fig.update_traces(marker=dict(size=6, opacity=0.75))
    fig.show()


## t-SNE — the settings Figure 4 panel f uses

In [ ]:
show(TSNE(n_components=2, perplexity=30, max_iter=1000, random_state=42).fit_transform(X),
     "t-SNE 1", "t-SNE 2", "t-SNE of cell morphology descriptors (perplexity 30)")
